# 09: Nerfstudio Integration for Aria Reconstruction

In this notebook we will demonstrate a reproducible pipeline to run a quick Nerf reconstruction using `nerfstudio`, export geometry (pointcloud and mesh), and transfer semantic labels (from SAM2 masks) onto the reconstructed geometry.

### In this notebook we will:
1. Setup & environment checks
2. Paths and dataset split selection
3. Export images, intrinsics and `transforms.json`
4. Hardware-safe nerfstudio config generator
5. Run short training and monitor metrics
6. Export geometry (pointcloud + mesh)
7. SAM2 mask export and per-image masks
8. Semantic projection onto geometry
9. Visualization and comparison with MPS pointcloud (notebook 08)

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks` .

## 9.1 Setup & Environment Checks

This section verifies the Python environment and GPU availability. It also computes a recommended downscale factor for images based on available GPU memory (VRAM) to avoid out-of-memory during quick experiments.

In [2]:
import sys
import platform
import subprocess

# Torch / CUDA checks
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('Torch CUDA available:', torch.cuda.is_available())
    print('Torch CUDA version:', torch.version.cuda)
except Exception as e:
    print('PyTorch not available or import failed:', e)

# Try to query nvidia-smi for GPU names and memory (MB)
def query_nvidia_smi():
    try:
        out = subprocess.check_output([
            'nvidia-smi',
            '--query-gpu=name,memory.total',
            '--format=csv,noheader,nounits'
        ], encoding='utf-8')
        lines = [l.strip() for l in out.strip().splitlines() if l.strip()]
        infos = []
        for l in lines:
            name, mem = [x.strip() for x in l.split(',')]
            infos.append({'name': name, 'memory_mb': int(mem)})
        return infos
    except Exception:
        return []

gpu_infos = query_nvidia_smi()
if len(gpu_infos) == 0:
    # Fallback to torch if available
    try:
        if torch.cuda.is_available():
            prop = torch.cuda.get_device_properties(0)
            gpu_infos = [{'name': prop.name, 'memory_mb': int(prop.total_memory // 1024 // 1024)}]
    except Exception:
        pass

if gpu_infos:
    for i, g in enumerate(gpu_infos):
        print(f"GPU {i}: {g['name']} — {g['memory_mb']} MB VRAM")
else:
    print('No GPU detected or nvidia-smi not available.')


PyTorch: 2.11.0+cu130
Torch CUDA available: True
Torch CUDA version: 13.0
GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU — 6144 MB VRAM


## 9.2 Paths & Dataset Split Selection

This section defines the directory structure and selects which frames to use for training and evaluation. We will choose a sparse subset of frames (~10% as eval set) to balance training speed with reconstruction quality.

In [16]:
import os
import numpy as np
import pandas as pd
from projectaria_tools.core import data_provider

# Define base directories
vrs_path = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'kettle_and_forklift_recording.vrs')
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
mps_dir = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Create output directory for nerfstudio dataset and training
nerfstudio_work_dir = os.path.join('..', 'data', 'outputs', 'nerfstudio', 'kettle_and_forklift')
os.makedirs(nerfstudio_work_dir, exist_ok=True)

# Subdirectories within nerfstudio work
dataset_dir = os.path.join(nerfstudio_work_dir, 'dataset')
images_dir = os.path.join(dataset_dir, 'images')
masks_export_dir = os.path.join(dataset_dir, 'masks')
training_dir = os.path.join(nerfstudio_work_dir, 'training')
exports_dir = os.path.join(nerfstudio_work_dir, 'exports')

# Create subdirectories
for d in [images_dir, masks_export_dir, training_dir, exports_dir]:
    os.makedirs(d, exist_ok=True)

print('Nerfstudio work directory:', nerfstudio_work_dir)
print('Dataset directory:', dataset_dir)
print('Training directory:', training_dir)
print('Exports directory:', exports_dir)

# Load VRS data provider to count available frames
provider = data_provider.create_vrs_data_provider(vrs_path)
stream_id = provider.get_stream_id_from_label('camera-rgb')
num_images = provider.get_num_data(stream_id)
print(f'\nTotal RGB frames available: {num_images}')

# load trajectory CSV from MPS export
trajectory_path = os.path.join(mps_dir, 'closed_loop_trajectory.csv')
trajectory_df = pd.read_csv(trajectory_path)
print(f'Trajectory loaded successfully! Found {len(trajectory_df)} poses.')

Nerfstudio work directory: ..\data\outputs\nerfstudio\kettle_and_forklift
Dataset directory: ..\data\outputs\nerfstudio\kettle_and_forklift\dataset
Training directory: ..\data\outputs\nerfstudio\kettle_and_forklift\training
Exports directory: ..\data\outputs\nerfstudio\kettle_and_forklift\exports

Total RGB frames available: 1551
Trajectory loaded successfully! Found 50626 poses.


In [25]:
# Select frame indices for training and evaluation
# Temporal downsampling: stride=3 selects one frame every 3 (~10 fps from 30 fps original)
all_frame_indices = np.arange(num_images, dtype=int)

stride = 3  # one every 3 frames
train_frames_candidate = all_frame_indices[::stride]

# For eval, pick additional frames not in train set (offset to increase viewpoint diversity)
eval_frames = all_frame_indices[1::stride]

# Limit eval set to ~10% of the total
target_eval_count = max(1, int(num_images * 0.1))
if len(eval_frames) > target_eval_count:
    # Randomly sample from eval candidates
    np.random.seed(42)
    eval_frames = np.random.choice(eval_frames, size=target_eval_count, replace=False)
    eval_frames = sorted(eval_frames)

train_frames = sorted([f for f in train_frames_candidate if f not in eval_frames])

print(f'Temporal downsampling: stride={stride} (one frame every 3)')
print(f'Selected {len(train_frames)} frames for training')
print(f'Selected {len(eval_frames)} frames for evaluation')
print(f'Eval frames: {eval_frames[:10]}... (showing first 10)')
print(f'Total frames in split: {len(train_frames) + len(eval_frames)}')

Temporal downsampling: stride=3 (one frame every 3)
Selected 517 frames for training
Selected 155 frames for evaluation
Eval frames: [np.int64(1), np.int64(7), np.int64(28), np.int64(31), np.int64(34), np.int64(46), np.int64(52), np.int64(55), np.int64(58), np.int64(67)]... (showing first 10)
Total frames in split: 672


## 9.3 Export Images & Compute transforms.json
In this section we will extract RGB images from VRS for selected frames, retrieve camera intrinsics and handle distortion correction, extract camera-to-world poses from VRS trajectory, and generate `transforms.json` in Nerfstudio format with intrinsics, poses, and frame metadata.


In [19]:
import os
import json
import cv2
import numpy as np
from scipy.spatial.transform import Rotation
from PIL import Image

# Retrieve RGB sensor calibration and extract camera-specific calibration
sensor_calib = provider.get_sensor_calibration(stream_id)
camera_calib = sensor_calib.camera_calibration()

# Get the correct label using the stream ID
rgb_label = provider.get_label_from_stream_id(stream_id)
print(f'RGB camera: {rgb_label}')

# Get calibration type
calib_type = camera_calib.get_model_name()
print(f'RGB calibration type: {calib_type}')

# Get image dimensions from the provider tuple
first_img_tuple = provider.get_image_data_by_index(stream_id, 0)
first_img = first_img_tuple[0].to_numpy_array()
img_height, img_width = first_img.shape[:2]
print(f'Image dimensions: {img_width} x {img_height}')

# Prepare frame list for transforms.json
frames_metadata = []
all_selected_frames = sorted(list(train_frames) + list(eval_frames))

print(f'\nExporting {len(all_selected_frames)} frames to {images_dir}...')

# Cache for intrinsics to avoid redundant computation
intrinsics_cached = None
distortion_params_cached = None

# Extrinsics: Device to Camera transform
T_Device_Camera = camera_calib.get_transform_device_camera().to_matrix()

for idx, frame_idx in enumerate(all_selected_frames):
    # Extract image data and timestamp
    img_tuple = provider.get_image_data_by_index(stream_id, frame_idx)
    img_array = img_tuple[0].to_numpy_array()
    timestamp_ns = img_tuple[1].capture_timestamp_ns
    
    # Save image
    img_filename = f'rgb_frame_{frame_idx:06d}.png'
    img_path = os.path.join(images_dir, img_filename)
    Image.fromarray(img_array).save(img_path)
    
    # Match the timestamp with the trajectory dataframe to get the world pose
    try:
        # convert trajectory timestamps to ns and find closest pose
        row_idx = (trajectory_df['tracking_timestamp_us'] * 1000 - timestamp_ns).abs().idxmin()
        pose_row = trajectory_df.loc[row_idx]
        
        tx = pose_row['tx_world_device']
        ty = pose_row['ty_world_device']
        tz = pose_row['tz_world_device']
        qw = pose_row['qw_world_device']
        qx = pose_row['qx_world_device']
        qy = pose_row['qy_world_device']
        qz = pose_row['qz_world_device']
        
        rot_matrix = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
        T_World_Device = np.eye(4)
        T_World_Device[:3, :3] = rot_matrix
        T_World_Device[:3, 3] = [tx, ty, tz]
        
        T_World_Camera = T_World_Device @ T_Device_Camera
        
    except Exception as e:
        print(f'  Error computing pose from trajectory for frame {frame_idx}: {e}')
        T_World_Camera = np.eye(4)
    
    # Get intrinsics
    if intrinsics_cached is None:
        try:
            proj_params = camera_calib.get_projection_params()
            
            fx, fy = proj_params[0], proj_params[1]
            cx, cy = proj_params[2], proj_params[3]
            
            distortion_params_cached = proj_params[4:] 
            print(f'Distortion detected: {calib_type}, coeffs shape: {len(distortion_params_cached)}')
            
            intrinsics_cached = {
                'fx': float(fx),
                'fy': float(fy),
                'cx': float(cx),
                'cy': float(cy)
            }
        except Exception as e:
            print(f'Error retrieving intrinsics: {e}')
            intrinsics_cached = {'fx': img_width, 'fy': img_height, 'cx': img_width / 2, 'cy': img_height / 2}
    
    # Build frame metadata
    frame_meta = {
        'file_path': f'images/{img_filename}',
        'transform_matrix': T_World_Camera.tolist(),
        'fl_x': intrinsics_cached['fx'],
        'fl_y': intrinsics_cached['fy'],
        'cx': intrinsics_cached['cx'],
        'cy': intrinsics_cached['cy'],
        'w': img_width,
        'h': img_height,
        'frame_index': int(frame_idx),
        'is_eval': int(frame_idx) in eval_frames
    }
    
    # Add distortion coefficients
    if distortion_params_cached is not None and len(distortion_params_cached) >= 4:
        frame_meta['k1'] = float(distortion_params_cached[0])
        frame_meta['k2'] = float(distortion_params_cached[1])
        frame_meta['p1'] = float(distortion_params_cached[2])
        frame_meta['p2'] = float(distortion_params_cached[3])
    
    frames_metadata.append(frame_meta)
    
    if (idx + 1) % 50 == 0:
        print(f'  Exported {idx + 1} / {len(all_selected_frames)} frames')

print(f'\nExported {len(all_selected_frames)} images successfully!')

RGB camera: camera-rgb
RGB calibration type: CameraModelType.FISHEYE624
Image dimensions: 1408 x 1408

Exporting 466 frames to ..\data\outputs\nerfstudio\kettle_and_forklift\dataset\images...
Distortion detected: CameraModelType.FISHEYE624, coeffs shape: 11
  Exported 50 / 466 frames
  Exported 100 / 466 frames
  Exported 150 / 466 frames
  Exported 200 / 466 frames
  Exported 250 / 466 frames
  Exported 300 / 466 frames
  Exported 350 / 466 frames
  Exported 400 / 466 frames
  Exported 450 / 466 frames

Exported 466 images successfully!


In [20]:
# Generate transforms.json for Nerfstudio
# Nerfstudio expects: "frames" array with file_path, transform_matrix (4x4), and optional intrinsics per frame

transforms_dict = {
    'camera_angle_x': 2 * np.arctan(img_width / (2 * intrinsics_cached['fx'])),  # Horizontal FoV in radians
    'frames': frames_metadata
}

# Save to JSON
transforms_json_path = os.path.join(dataset_dir, 'transforms.json')
with open(transforms_json_path, 'w') as f:
    json.dump(transforms_dict, f, indent=2)

print(f'transforms.json saved to {transforms_json_path}')
print(f'Total frames in transforms.json: {len(frames_metadata)}')

# Summary statistics
eval_count = sum(1 for fm in frames_metadata if fm['is_eval'])
train_count = len(frames_metadata) - eval_count
print(f'  Train frames: {train_count}')
print(f'  Eval frames: {eval_count}')
print(f'  Train/Eval split: {train_count / len(frames_metadata) * 100:.1f}% / {eval_count / len(frames_metadata) * 100:.1f}%')

transforms.json saved to ..\data\outputs\nerfstudio\kettle_and_forklift\dataset\transforms.json
Total frames in transforms.json: 466
  Train frames: 311
  Eval frames: 155
  Train/Eval split: 66.7% / 33.3%


## 9.4 Hardware-Safe Nerfstudio Config Generator

We will generate a nerfstudio training configuration (YAML) that adapts to available GPU (6 GB VRAM). We wil set batch size and model parameters conservatively and configure training iterations then save config to `nerfstudio_config.yaml` in the dataset directory


In [ ]:
# GPU configuration: keep original image resolution, but use temporal downsampling (stride=3)
# User requested original image quality; this keeps full resolution but fewer frames.

gpu_vram_gb = 6.0
recommended_scale = 1.0  # keep original resolution

print(f'GPU VRAM: {gpu_vram_gb:.1f} GB')
print(f'Spatial downscale: {recommended_scale:.2f}x (keep original image quality)')
print('Temporal downsampling: 1 frame every 3 (configured in section 9.2)')

# Compute actual resolution (original)
scaled_width = int(img_width * recommended_scale)
scaled_height = int(img_height * recommended_scale)
print(f'Training resolution: {scaled_width} x {scaled_height}')

# Batch size for 6GB VRAM: keep small to avoid OOM when using full resolution
train_batch_size = 6
eval_batch_size = 2

print(f'Train batch size: {train_batch_size}')
print(f'Eval batch size: {eval_batch_size}')

# Training iterations: quick demo (3k iters)
num_iterations = 3000
steps_per_eval = 250
steps_per_save = 500

print(f'Training iterations: {num_iterations}')
print(f'Steps per eval: {steps_per_eval}')
print(f'Steps per save: {steps_per_save}')

GPU VRAM: 6.0 GB
Image downscale: 0.50x (conservative for 6GB VRAM)
Training resolution: 704 x 704
Train batch size: 8
Eval batch size: 4
Training iterations: 3000
Steps per eval: 250
Steps per save: 500


In [24]:
# Generate nerfstudio config YAML
# Nerfstudio uses YAML config files to specify training parameters, model, and data settings

nerfstudio_config_yaml = f"""# Nerfstudio Config for Aria Kettle & Forklift Dataset
# Generated for 6GB GPU with 0.5x downscale

method-name: nerfacto
experiment-name: aria_kettle_forklift_0p5x

pipeline:
  datamanager:
    train-num-rays-per-batch: {train_batch_size}
    eval-num-rays-per-batch: {eval_batch_size}
    camera-optimizer:
      mode: off
  model:
    eval-num-rays-per-chunk: 8192

optimizers:
  fields:
    optimizer:
      type: adam
      lr: 1.0e-2
    scheduler:
      type: exponential
      lr-final: 1.0e-4
      max-steps: {num_iterations}
  camera-opt:
    optimizer:
      type: adam
      lr: 1.0e-3
    scheduler:
      type: constant

viewer:
  quit-on-train-completion: false

vis: tensorboard

logging:
  local-runs-dir: {training_dir}

steps-per-save: {steps_per_save}
steps-per-eval-image: {steps_per_eval}
steps-per-eval-batch: {steps_per_eval}
max-num-iterations: {num_iterations}
mixed-precision: false
"""

config_path = os.path.join(dataset_dir, 'nerfstudio_config.yaml')
with open(config_path, 'w') as f:
    f.write(nerfstudio_config_yaml)

print(f'Nerfstudio config saved to {config_path}')
print(f'\nConfig summary:')
print(f'  Model: nerfacto')
print(f'  Train batch size: {train_batch_size}')
print(f'  Eval batch size: {eval_batch_size}')
print(f'  Image downscale: {recommended_scale}x')
print(f'  Max iterations: {num_iterations}')
print(f'  Mixed precision: false (for stability on 6GB VRAM)')


Nerfstudio config saved to ..\data\outputs\nerfstudio\kettle_and_forklift\dataset\nerfstudio_config.yaml

Config summary:
  Model: nerfacto
  Train batch size: 8
  Eval batch size: 4
  Image downscale: 0.5x
  Max iterations: 3000
  Mixed precision: false (for stability on 6GB VRAM)
